# 04 · Pre-exam practice features

For every student and midterm, summarize the practice they did **before** that exam. Two leakage guards:

1. Every event inside **any** of the student's exam windows is removed first (so midterm 1 answers are never midterm 2 practice).
2. Only events before `exam_start - cutoff_days` are used. `cutoff_days = 0` is the main feature set; 3, 7, 14, 21 and 28 feed the early-warning curve in notebook 06.

Two engines build complementary features: pandas (per-problem quality: first try, best, gain) and DuckDB SQL
(coverage, timing, consistency). Both apply the same guards; the SQL version is shown in full below.

In [1]:
import pandas as pd
from so_ell import config
from so_ell.notebook import show, show_sql, describe
pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 200)
config.ensure_dirs()
T = config.TABLES_DIR

## 1. Practice events with exam windows removed

In [2]:
from so_ell import features as F, scores, sql_features, validation
from so_ell.config import WINDOW_KEYS
show(F.drop_exam_window_events, F.score_study_events)
windows = scores.build_windows(pd.read_parquet(config.EXAM_ATTEMPTS_PARQUET))
student_scores = pd.read_parquet(config.PROCESSED_DIR / "student_exam_scores.parquet")
raw = scores.load_scoreable_events()
study = F.score_study_events(raw, windows)
print(f"{len(raw):,} answer events -> {len(study):,} outside every exam window")
del raw

`so_ell.features.drop_exam_window_events`
```python
def drop_exam_window_events(events: pd.DataFrame, windows: pd.DataFrame) -> pd.DataFrame:
    """Remove every event that falls inside any of that student's exam windows."""
    events = events.reset_index(drop=True)
    probe = events[["semester_raw", "anon_student_id", "time"]].reset_index().merge(
        windows[["semester_raw", "anon_student_id", "exam_start", "exam_end"]],
        on=["semester_raw", "anon_student_id"],
        how="inner",
    )
    inside = probe.loc[probe["time"].between(probe["exam_start"], probe["exam_end"], inclusive="both"), "index"]
    return events.loc[~events.index.isin(inside.unique())].copy()
```

`so_ell.features.score_study_events`
```python
def score_study_events(raw: pd.DataFrame, windows: pd.DataFrame) -> pd.DataFrame:
    """Practice events (exam windows removed) with per-event scores attached."""
    study = add_event_scores(raw)
    study = drop_exam_window_events(study, windows)
    keys = infer_answer_keys(study)
    study = study.merge(keys[["problem_name", "answer_key"]], on="problem_name", how="left")
    keyed = study["family"].isin(KEYED_FAMILIES) & study["response_norm"].notna()
    study.loc[keyed, "event_score"] = (study.loc[keyed, "response_norm"] == study.loc[keyed, "answer_key"]).astype(float)
    return study
```

16,764,045 answer events -> 14,182,560 outside every exam window


## 2. Pandas features: per-problem quality

In [3]:
show(F.summarize_problems, F.build_practice_features)
practice = F.build_practice_features(study, windows, student_scores, cutoff_days=0)
practice.shape

`so_ell.features.summarize_problems`
```python
def summarize_problems(problem_pre: pd.DataFrame) -> pd.DataFrame:
    """First / last / best score per (student, exam, family, problem).

    groupby.first/last skip NaN, so "first" is the first *scored* attempt.
    Opening a Parsons / ActiveCode problem and never getting a score counts as 0.
    """
    keys = WINDOW_KEYS + ["family", "problem_name"]
    ordered = problem_pre.sort_values(keys + ["time"])
    g = ordered.groupby(keys, observed=True)["event_score"]
    out = pd.DataFrame(
        {
            "problem_events": g.size(),
            "first_score": g.first(),
            "last_score": g.last(),
            "best_score": g.max(),
        }
    ).reset_index()
    unscored = out["best_score"].isna() & out["family"].isin(["parsons", "activecode"])
    out.loc[unscored, ["first_score", "last_score", "best_score"]] = 0.0
    out["score_gain"] = out["last_score"] - out["first_score"]
    return out
```

`so_ell.features.build_practice_features`
```python
def build_practice_features(
    study: pd.DataFrame,
    windows: pd.DataFrame,
    student_scores: pd.DataFrame,
    cutoff_days: int = 0,
) -> pd.DataFrame:
    pre = study[["semester_raw", "anon_student_id", "time", "problem_name", "family", "event_score"]].merge(
        windows[WINDOW_KEYS + ["exam_start"]], on=["semester_raw", "anon_student_id"], how="inner"
    )
    cutoff = pre["exam_start"] - pd.to_timedelta(cutoff_days, unit="D")
    pre = pre.loc[pre["time"] < cutoff].copy()
    pre["event_day"] = pre["time"].dt.floor("D")

    def activity(frame: pd.DataFrame, by: list[str]) -> pd.DataFrame:
        return frame.groupby(by, as_index=False, observed=True).agg(
            total_events=("problem_name", "size"),
            active_days=("event_day", "nunique"),
            unique_problems=("problem_name", "nunique"),
            scored_events=("event_score", lambda s: s.notna().sum()),
            avg_event_score=("event_score", "mean"),
        )

    # conceptcheck (selectquestion) never carries a score, so its avg is all-NaN by design.
    row_activity = pd.concat(
        [activity(pre, WINDOW_KEYS + ["family"]), activity(pre, WINDOW_KEYS).assign(family="all_activity")],
        ignore_index=True,
    )

    per_problem = summarize_problems(pre.loc[pre["family"].isin(SCOREABLE_FAMILIES)])

    def quality(frame: pd.DataFrame, by: list[str]) -> pd.DataFrame:
        return frame.groupby(by, as_index=False, observed=True).agg(
            problems_seen=("problem_name", "nunique"),
            scored_problems=("best_score", lambda s: s.notna().sum()),
            mean_first_score=("first_score", "mean"),
            mean_last_score=("last_score", "mean"),
            mean_best_score=("best_score", "mean"),
            mean_score_gain=("score_gain", "mean"),
            mean_events_per_problem=("problem_events", "mean"),
        )

    problem_quality = pd.concat(
        [quality(per_problem, WINDOW_KEYS + ["family"]), quality(per_problem, WINDOW_KEYS).assign(family="learning")],
        ignore_index=True,
    )

    features = student_scores.merge(_wide(row_activity), on=WINDOW_KEYS, how="left").merge(
        _wide(problem_quality), on=WINDOW_KEYS, how="left"
    )
    # A student with no practice of a type did zero events of it (not "unknown").
    count_cols = [c for c in features.columns if c.endswith(("__total_events", "__active_days", "__unique_problems", "__scored_events", "__problems_seen", "__scored_problems"))]
    features[count_cols] = features[count_cols].fillna(0)
    features["cohort"] = features["semester_raw"].astype(str) + "_" + features["midterm"].astype(str)
    features["cutoff_days"] = cutoff_days
    return features
```

(2745, 110)

## 3. SQL features (DuckDB)
The event log is queried in place; these files are executed in order.

In [4]:
show(sql_features.connect, sql_features.build_sql_features)
show_sql("04_pre_exam_events.sql", "05_student_exam_features.sql", "06_chapter_diversity.sql", "08_timing_bins.sql")
sql = sql_features.build_sql_features(cutoff_days=0)
sql["student_features"].shape

`so_ell.sql_features.connect`
```python
def connect(
    events_path: Path = config.EVENTS_PARQUET,
    attempts_path: Path = config.EXAM_ATTEMPTS_PARQUET,
) -> duckdb.DuckDBPyConnection:
    if not events_path.exists():
        raise FileNotFoundError(
            f"Missing {events_path}. Run `so-ell ingest` first."
        )
    if not attempts_path.exists():
        raise FileNotFoundError(
            f"Missing {attempts_path}. Run `so-ell exam-windows` first."
        )

    con = duckdb.connect()
    # 20M rows is fine on a laptop, but no reason to use a single thread.
    con.execute("PRAGMA threads=4")
    con.execute("SET enable_progress_bar = false")
    semesters = ", ".join(f"'{s}'" for s in config.ANALYSIS_SEMESTERS)
    con.execute(
        f"CREATE OR REPLACE VIEW raw_events AS SELECT * FROM read_parquet('{_sql_path(events_path)}') "
        f"WHERE semester IN ({semesters})"
    )
    con.execute(
        f"CREATE OR REPLACE VIEW exam_attempts AS SELECT * FROM read_parquet('{_sql_path(attempts_path)}')"
    )
    return con
```

`so_ell.sql_features.build_sql_features`
```python
def build_sql_features(
    con: duckdb.DuckDBPyConnection | None = None, cutoff_days: int = 0
) -> dict[str, pd.DataFrame]:
    """Run the SQL feature queries and return named frames."""
    close_when_done = False
    if con is None:
        con = connect()
        close_when_done = True
    try:
        con.execute(f"SET VARIABLE cutoff_days = {int(cutoff_days)}")
        overview = query(con, "01_event_overview.sql")
        by_semester = query(con, "02_events_by_semester.sql")
        windows = query(con, "03_exam_windows.sql")
        pre_count = run_pre_exam_views(con)
        student_features = query(con, "05_student_exam_features.sql")
        diversity = query(con, "06_chapter_diversity.sql")
        chapter_cohort = query(con, "07_chapter_by_cohort.sql")
        timing_bins = query(con, "08_timing_bins.sql")
        family_volume = query(con, "09_family_volume.sql")

        student_features = student_features.merge(
            diversity,
            on=["semester_raw", "anon_student_id", "midterm"],
            how="left",
        )
        return {
            "overview": overview,
            "by_semester": by_semester,
            "windows": windows,
            "pre_count": pre_count,
            "student_features": student_features,
            "chapter_cohort": chapter_cohort,
            "timing_bins": timing_bins,
            "family_volume": family_volume,
        }
    finally:
        if close_when_done:
            con.close()
```

`sql/04_pre_exam_events.sql`
```sql
-- Pre-exam practice events joined onto each student's exam window.
--
-- Leakage guards:
--   1. Only events strictly before this exam's start.
--   2. Events inside ANY of the student's timed exam windows are dropped
--      (the practice_events view). Without this, a student's midterm 1 answers
--      would count as "practice" for midterm 2. The first version of this
--      file only had guard 1.
-- days_before is 1 on the day before the exam, 7 a week before, etc.
-- Optional: SET VARIABLE cutoff_days = 14; keeps only practice at least
-- 14 days before the exam (used by the early-warning analysis).

CREATE OR REPLACE VIEW exam_windows AS
SELECT
    semester_raw,
    anon_student_id,
    midterm,
    ANY_VALUE(semester) AS semester,
    MIN(attempt_start) AS exam_start,
    MAX(attempt_end) AS exam_end
FROM exam_attempts
GROUP BY semester_raw, anon_student_id, midterm;

CREATE OR REPLACE VIEW practice_events AS
SELECT e.*
FROM raw_events e
WHERE e.time IS NOT NULL
  AND NOT EXISTS (
      SELECT 1
      FROM exam_windows x
      WHERE x.semester_raw = e.semester
        AND x.anon_student_id = e.anon_student_id
        AND e.time BETWEEN x.exam_start AND x.exam_end
  );

CREATE OR REPLACE VIEW pre_exam_events AS
SELECT
    e.semester AS semester_raw,
    e.anon_student_id,
    w.midterm,
    e.time,
    CAST(e.time AS DATE) AS event_day,
    e.session_id,
    e.problem_name,
    e.level_chapter,
    e.level_subchapter,
    e.selection,
    e.cf_week_no,
    DATE_DIFF('day', CAST(e.time AS DATE), CAST(w.exam_start AS DATE)) AS days_before,
    CASE e.selection
        WHEN 'mChoice' THEN 'mchoice'
        WHEN 'parsonsMove' THEN 'parsons'
        WHEN 'parsons' THEN 'parsons'
        WHEN 'hparsons' THEN 'parsons'
        WHEN 'hparsonsAnswer' THEN 'parsons'
        WHEN 'activecode' THEN 'activecode'
        WHEN 'unittest' THEN 'activecode'
        WHEN 'ac_error' THEN 'activecode'
        WHEN 'fillb' THEN 'fillb'
        WHEN 'clickableArea' THEN 'clickable'
        WHEN 'dragNdrop' THEN 'dragndrop'
        WHEN 'dragNdrop-drop' THEN 'dragndrop'
        WHEN 'shortanswer' THEN 'shortanswer'
        WHEN 'selectquestion' THEN 'conceptcheck'
        ELSE 'other'
    END AS family
FROM practice_events e
INNER JOIN exam_windows w
    ON e.semester = w.semester_raw
   AND e.anon_student_id = w.anon_student_id
WHERE e.time < w.exam_start - to_days(CAST(COALESCE(getvariable('cutoff_days'), 0) AS INTEGER))
  AND e.selection IN (
      'mChoice', 'fillb', 'parsonsMove', 'parsons', 'activecode',
      'unittest', 'ac_error', 'hparsons', 'hparsonsAnswer',
      'clickableArea', 'dragNdrop', 'dragNdrop-drop',
      'shortanswer', 'selectquestion'
  );

SELECT COUNT(*) AS n_pre_exam_events
FROM pre_exam_events;
```

`sql/05_student_exam_features.sql`
```sql
-- Student-exam features that are easier in SQL than in pandas:
-- topic coverage, spacing / cramming, session mix, activity mix.
--
-- Column names use family__metric so they drop into the existing
-- within-cohort correlation code.

SELECT
    semester_raw,
    anon_student_id,
    midterm,

    COUNT(*) AS coverage__total_events,
    COUNT(DISTINCT problem_name) AS coverage__unique_problems,
    COUNT(DISTINCT CASE
        WHEN level_chapter IS NOT NULL
         AND TRIM(level_chapter) NOT IN ('', 'py4e-int', 'ack', 'toctree', 'index')
        THEN level_chapter
    END) AS coverage__unique_chapters,
    COUNT(DISTINCT CASE
        WHEN level_subchapter IS NOT NULL
         AND TRIM(level_subchapter) NOT IN ('', 'index', 'toctree')
        THEN level_subchapter
    END) AS coverage__unique_subchapters,
    COUNT(DISTINCT family) AS coverage__family_count,

    COUNT(DISTINCT event_day) AS consistency__active_days,
    COUNT(DISTINCT session_id) AS consistency__unique_sessions,
    COUNT(DISTINCT cf_week_no) AS consistency__unique_weeks,
    DATE_DIFF('day', MIN(event_day), MAX(event_day)) AS consistency__span_days,
    COUNT(DISTINCT problem_name)::DOUBLE
        / NULLIF(COUNT(DISTINCT event_day), 0) AS consistency__problems_per_active_day,
    COUNT(*)::DOUBLE
        / NULLIF(COUNT(DISTINCT event_day), 0) AS consistency__events_per_active_day,

    MAX(days_before) AS timing__first_days_before,
    MIN(days_before) AS timing__last_days_before,
    SUM(CASE WHEN days_before BETWEEN 1 AND 3 THEN 1 ELSE 0 END) AS timing__events_last_3d,
    SUM(CASE WHEN days_before BETWEEN 1 AND 7 THEN 1 ELSE 0 END) AS timing__events_last_7d,
    SUM(CASE WHEN days_before BETWEEN 1 AND 14 THEN 1 ELSE 0 END) AS timing__events_last_14d,
    SUM(CASE WHEN days_before BETWEEN 1 AND 28 THEN 1 ELSE 0 END) AS timing__events_last_28d,
    SUM(CASE WHEN days_before BETWEEN 1 AND 3 THEN 1 ELSE 0 END)::DOUBLE
        / NULLIF(COUNT(*), 0) AS timing__share_last_3d,
    SUM(CASE WHEN days_before BETWEEN 1 AND 7 THEN 1 ELSE 0 END)::DOUBLE
        / NULLIF(COUNT(*), 0) AS timing__share_last_7d,
    SUM(CASE WHEN days_before BETWEEN 1 AND 14 THEN 1 ELSE 0 END)::DOUBLE
        / NULLIF(COUNT(*), 0) AS timing__share_last_14d,

    SUM(CASE WHEN family = 'parsons' THEN 1 ELSE 0 END)::DOUBLE
        / NULLIF(COUNT(*), 0) AS mix__parsons_share,
    SUM(CASE WHEN family = 'activecode' THEN 1 ELSE 0 END)::DOUBLE
        / NULLIF(COUNT(*), 0) AS mix__activecode_share,
    SUM(CASE WHEN family IN ('parsons', 'activecode') THEN 1 ELSE 0 END)::DOUBLE
        / NULLIF(COUNT(*), 0) AS mix__coding_share,
    SUM(CASE WHEN family = 'mchoice' THEN 1 ELSE 0 END)::DOUBLE
        / NULLIF(COUNT(*), 0) AS mix__mchoice_share,
    SUM(CASE WHEN family = 'conceptcheck' THEN 1 ELSE 0 END)::DOUBLE
        / NULLIF(COUNT(*), 0) AS mix__conceptcheck_share
FROM pre_exam_events
GROUP BY semester_raw, anon_student_id, midterm;
```

`sql/06_chapter_diversity.sql`
```sql
-- How spread out was a student's practice across chapters?
-- 1 - Herfindahl, so 0 = all events in one chapter, closer to 1 = more even.

WITH chapter_counts AS (
    SELECT
        semester_raw,
        anon_student_id,
        midterm,
        level_chapter,
        COUNT(*) AS n_events
    FROM pre_exam_events
    WHERE level_chapter IS NOT NULL
      AND TRIM(level_chapter) NOT IN ('', 'py4e-int', 'ack', 'toctree', 'index')
    GROUP BY 1, 2, 3, 4
),
with_share AS (
    SELECT
        *,
        n_events / SUM(n_events) OVER (
            PARTITION BY semester_raw, anon_student_id, midterm
        ) AS chapter_share
    FROM chapter_counts
)
SELECT
    semester_raw,
    anon_student_id,
    midterm,
    1 - SUM(chapter_share * chapter_share) AS coverage__chapter_diversity
FROM with_share
GROUP BY semester_raw, anon_student_id, midterm;
```

`sql/08_timing_bins.sql`
```sql
-- Last-28-day timing bins, still at the student-exam grain.
-- We collapse this to score bands later so Looker doesn't need student ids.

SELECT
    semester_raw,
    anon_student_id,
    midterm,
    CASE
        WHEN days_before BETWEEN 1 AND 3 THEN '1-3 days before'
        WHEN days_before BETWEEN 4 AND 7 THEN '4-7 days before'
        WHEN days_before BETWEEN 8 AND 14 THEN '8-14 days before'
        WHEN days_before BETWEEN 15 AND 28 THEN '15-28 days before'
    END AS timing_bin,
    COUNT(*) AS n_events
FROM pre_exam_events
WHERE days_before BETWEEN 1 AND 28
GROUP BY 1, 2, 3, 4;
```

(2775, 29)

## 4. Merge, validate, save

In [5]:
from so_ell import pipeline
extra = [c for c in sql["student_features"].columns if c not in WINDOW_KEYS]
features = practice.merge(sql["student_features"][WINDOW_KEYS + extra], on=WINDOW_KEYS, how="left")
validation.validate(features, validation.features_schema)
features.to_parquet(config.PROCESSED_DIR / "student_exam_features.parquet", index=False)
sql["overview"].to_csv(T / "event_log_overview.csv", index=False)
sql["by_semester"].to_csv(T / "events_by_semester.csv", index=False)
sql["chapter_cohort"].to_csv(T / "chapter_activity_by_cohort.csv", index=False)
pipeline.public_timing_table(sql["timing_bins"], features).to_csv(T / "practice_timing_by_score_band.csv", index=False)
print(f"{features.shape[0]:,} student-exams x {features.shape[1]} columns")

2,745 student-exams x 136 columns


Feature families (column prefix before `__`):

In [6]:
fam = pd.Series([c.split("__")[0] for c in features.columns if "__" in c]).value_counts()
fam.to_frame("columns")

,columns
activecode,12
clickable,12
dragndrop,12
fillb,12
mchoice,12
parsons,12
timing,9
learning,7
coverage,6
consistency,6


Summary of the main features (aggregate only). The full list with definitions is in `docs/feature_dictionary.md`.

In [7]:
from so_ell.models import PRACTICE_FEATURES
describe(features, [c for c in PRACTICE_FEATURES if c in features])

,count,mean,std,min,25%,50%,75%,max
parsons__mean_first_score,2739.0,0.368,0.169,0.000,0.244,0.360,0.483,1.000
mchoice__mean_first_score,2741.0,0.510,0.115,0.000,0.429,0.515,0.590,0.892
activecode__mean_best_score,2739.0,0.269,0.091,0.000,0.207,0.257,0.320,0.950
all_activity__total_events,2745.0,7459.673,7956.875,0.000,3210.000,4903.000,7836.000,91818.000
all_activity__active_days,2745.0,22.889,9.828,0.000,16.000,22.000,30.000,60.000
coverage__unique_chapters,2741.0,18.628,2.640,2.000,17.000,19.000,21.000,24.000
coverage__chapter_diversity,2741.0,0.866,0.056,0.219,0.850,0.880,0.899,0.928
timing__share_last_7d,2741.0,0.218,0.156,0.000,0.114,0.186,0.288,1.000
consistency__problems_per_active_day,2741.0,25.840,8.032,2.000,20.714,24.381,29.333,127.333
mix__coding_share,2741.0,0.630,0.067,0.000,0.601,0.637,0.668,0.845


## 5. Leakage check
No practice event may sit inside any exam window. Count events in `study` that fall inside one of the student's windows (should be 0), versus the same count on the unfiltered answer events. The synthetic end-to-end test (`tests/test_pipeline_end_to_end.py`) asserts the same thing for both feature engines.

In [8]:
def events_inside_windows(events):
    probe = events[["semester_raw", "anon_student_id", "time"]].merge(
        windows[["semester_raw", "anon_student_id", "exam_start", "exam_end"]], on=["semester_raw", "anon_student_id"]
    )
    return int(probe["time"].between(probe["exam_start"], probe["exam_end"]).sum())

print("answer events inside an exam window, before the guard:", f"{events_inside_windows(scores.load_scoreable_events()):,}")
print("practice events inside an exam window, after the guard:", f"{events_inside_windows(study):,}")

answer events inside an exam window, before the guard: 2,581,485


practice events inside an exam window, after the guard: 0


## 6. Early-warning feature sets
The same code with the line moved N days before the exam.

In [9]:
for cutoff in [c for c in pipeline.LEAD_TIME_CUTOFFS if c]:
    early = F.build_practice_features(study, windows, student_scores, cutoff_days=cutoff)
    early_sql = sql_features.build_sql_features(cutoff_days=cutoff)["student_features"]
    early = early.merge(early_sql[WINDOW_KEYS + [c for c in early_sql.columns if c not in WINDOW_KEYS]], on=WINDOW_KEYS, how="left")
    early.to_parquet(config.PROCESSED_DIR / f"student_exam_features_cutoff_{cutoff}d.parquet", index=False)
    print(f"cutoff {cutoff:>2}d: mean events per student-exam {early['all_activity__total_events'].mean():7.1f}")

cutoff 28d: mean events per student-exam  3705.6


cutoff 21d: mean events per student-exam  4572.5


cutoff 14d: mean events per student-exam  5110.0


cutoff  7d: mean events per student-exam  5800.1


cutoff  3d: mean events per student-exam  6262.7


In [10]:
del study